# Projekt Dyplomowy: Diagnostyka nowotworów z danych medycznych
### Autor: Krzysztof Ledworowski
**Kurs: Data Scientist Online + AI**

Niniejszy notebook realizuje projekt dyplomowy end-to-end zgodnie z wytycznymi z Lekcji 42.
Obejmuje definicję problemu, EDA, pipeline scikit-learn, porównanie modeli (Baseline vs 3 algorytmy),
analizę błędów, ewaluację Green IT oraz automatyczne wygenerowanie struktury plików z aplikacją FastAPI i Dockerfile.

## 1. Instalacja zależności
Uruchom poniższą komórkę, aby upewnić się, że wszystkie wymagane biblioteki są zainstalowane w środowisku.

In [ ]:
# Instalujemy wymagane pakiety:
# - scikit-learn: algorytmy uczenia maszynowego i wbudowane zbiory danych
# - pandas: manipulacja danymi tabelarycznymi
# - numpy: obliczenia macierzowe i numeryczne
# - matplotlib i seaborn: generowanie wykresów i wizualizacja
# - fastapi i uvicorn: budowa i uruchomienie serwera API do wdrożenia
# - joblib: zapis i odczyt wytrenowanego modelu
# - pydantic: walidacja typów danych wejściowych do API
!pip install scikit-learn pandas numpy matplotlib seaborn fastapi uvicorn joblib pydantic

## 2. Import bibliotek i konfiguracja reprodukowalności

In [ ]:
# Importujemy moduł do operacji na plikach i katalogach systemowych
import os

# Importujemy moduł do pomiaru czasu wykonania (niezbędny do analizy Green IT)
import time

# Importujemy moduł do pracy z formatem JSON (zapis metadanych modelu)
import json

# Importujemy pandas do operacji na ramkach danych
import pandas as pd

# Importujemy numpy do operacji wektorowych i matematycznych
import numpy as np

# Importujemy biblioteki do tworzenia wykresów
import matplotlib.pyplot as plt
import seaborn as sns

# Importujemy wbudowany medyczny zbiór danych o nowotworach
from sklearn.datasets import load_breast_cancer

# Importujemy funkcję do podziału danych na zbiór treningowy i testowy oraz do walidacji krzyżowej
from sklearn.model_selection import train_test_split, cross_val_score

# Importujemy narzędzia do budowy bezpiecznego potoku przetwarzania (brak data leakage)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Importujemy model bazowy (DummyClassifier) jako punkt odniesienia
from sklearn.dummy import DummyClassifier

# Importujemy 3 algorytmy klasyfikacyjne do porównania
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Importujemy metryki oceny jakości modeli
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Importujemy joblib do serializacji (zapisania na dysku) modelu
import joblib

# Ustawiamy globalną stałą ziarna losowości, aby wyniki były w 100% powtarzalne
RANDOM_STATE = 42

# Ustawiamy styl wykresów na czytelny i estetyczny
sns.set_theme(style="whitegrid")
print("Zaimportowano biblioteki i ustawiono ziarno losowości: RANDOM_STATE = 42")

## 3. Definicja problemu i metryka sukcesu

In [ ]:
# Tworzymy słownik opisujący założenia biznesowo-medyczne projektu
project_info = {
    "autor": "Krzysztof Ledworowski",
    "problem": "Wczesna klasyfikacja guzów piersi na łagodne (benign) i złośliwe (malignant)",
    "zrodlo_danych": "Breast Cancer Wisconsin Diagnostic Dataset (scikit-learn)",
    "metryka_sukcesu": "Accuracy (dokładność ogólna) > 90%",
    "cel_wdrozenia": "Mikroserwis REST API (FastAPI) w kontenerze Docker"
}

# Wyświetlamy sformatowane założenia projektu
for k, v in project_info.items():
    print(f"{k.upper()}: {v}")

## 4. Pobranie danych i wstępna eksploracja (EDA)

In [ ]:
# Wczytujemy surowe dane medyczne z biblioteki scikit-learn
raw_data = load_breast_cancer()

# Tworzymy ramkę danych (DataFrame) zawierającą 30 cech jądra komórkowego
df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)

# Dodajemy kolumnę celu (target): 0 oznacza guz złośliwy, 1 oznacza guz łagodny
df['target'] = raw_data.target

# Wyświetlamy wymiary zbioru (liczbę wierszy i kolumn)
print(f"Wymiary zbioru danych: {df.shape[0]} wierszy, {df.shape[1]} kolumn.")

# Sprawdzamy, czy w danych występują jakiekolwiek braki danych (NaN)
braki = df.isnull().sum().sum()
print(f"Suma braków danych (NaN): {braki}")

# Wyświetlamy rozkład klas decyzyjnych w zbiorze
print("\nRozkład klas (0 = Złośliwy, 1 = Łagodny):")
print(df['target'].value_counts())

# Wyświetlamy 5 pierwszych wierszy zbioru danych
df.head()

## 5. Wizualizacja rozkładu klas i korelacji

In [ ]:
# Inicjalizujemy okno wykresu o określonych wymiarach
plt.figure(figsize=(14, 5))

# Tworzymy pierwszy podwykres: rozkład klas w zbiorze danych
plt.subplot(1, 2, 1)
# Rysujemy wykres słupkowy częstości występowania obu klas
sns.countplot(x='target', data=df, hue='target', palette='pastel', legend=False)
# Ustawiamy tytuł wykresu
plt.title('Rozkład klas nowotworu (0 = Złośliwy, 1 = Łagodny)')
# Podpisujemy oś X
plt.xlabel('Klasa')
# Podpisujemy oś Y
plt.ylabel('Liczba próbek')

# Tworzymy drugi podwykres: zależność między promienistym rozmiarem a wklęsłością jądra
plt.subplot(1, 2, 2)
# Rysujemy wykres punktowy z rozróżnieniem kolorystycznym klas
sns.scatterplot(
    x='mean radius', 
    y='mean concave points', 
    hue='target', 
    data=df, 
    palette='coolwarm', 
    alpha=0.8
)
# Ustawiamy tytuł drugiego wykresu
plt.title('Średni promień vs Średnie punkty wklęsłe')

# Dopasowujemy marginesy między wykresami
plt.tight_layout()
# Wyświetlamy wykresy w notebooku
plt.show()

## 6. Podział danych na zbiór treningowy i testowy

In [ ]:
# Wydzielamy macierz cech wejściowych X (usuwamy kolumnę celu 'target')
X = df.drop(columns=['target'])

# Wydzielamy wektor etykiet y (sama kolumna 'target')
y = df['target']

# Dzielimy dane na zbiór treningowy (80%) i testowy (20%)
# stratify=y gwarantuje, że proporcja klas w obu zbiorach będzie identyczna
# random_state zapewnia pełną powtarzalność losowania
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=RANDOM_STATE, 
    stratify=y
)

# Wyświetlamy liczbę próbek w każdym ze zbiorów
print(f"Liczba próbek treningowych: {X_train.shape[0]}")
print(f"Liczba próbek testowych: {X_test.shape[0]}")

## 7. Definicja potoków (Pipelines) dla modeli

In [ ]:
# Tworzymy słownik z 4 modelami do przetestowania.
# Każdy model zamykamy w obiekcie Pipeline z StandardScalerem.
# Dzięki temu normalizacja dopasowuje się TYLKO na zbiorze treningowym (brak Data Leakage).
models = {
    # 1. Model bazowy - typuje klasę większościową (punkt odniesienia)
    "Baseline (Dummy)": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", DummyClassifier(strategy="most_frequent"))
    ]),
    # 2. Regresja Logistyczna - szybki, prosty model liniowy
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(random_state=RANDOM_STATE, max_iter=1000))
    ]),
    # 3. Random Forest - las drzew decyzyjnych
    "Random Forest": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
    ]),
    # 4. Gradient Boosting - sekwencyjne wzmacnianie drzew
    "Gradient Boosting": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE))
    ])
}
print("Zdefiniowano potoki dla modeli: Baseline, Logistic Regression, Random Forest, Gradient Boosting.")

## 8. Trening, ewaluacja CV oraz analiza Green IT

In [ ]:
# Tworzymy słownik, w którym zapiszemy wyniki metryk i pomiarów czasu
results = {}

# Iterujemy po każdym zdefiniowanym modelu
for name, pipe in models.items():
    # Pobieramy dokładny czas przed rozpoczęciem treningu
    start_time = time.perf_counter()
    
    # Trenujemy cały potok (skaler + klasyfikator) na danych treningowych
    pipe.fit(X_train, y_train)
    
    # Obliczamy czas trwania treningu w sekundach
    train_time = time.perf_counter() - start_time
    
    # Wykonujemy 5-krotną walidację krzyżową na zbiorze treningowym
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    
    # Generujemy predykcje na odłożonym, niewidzianym zbiorze testowym
    y_pred = pipe.predict(X_test)
    
    # Liczymy dokładność (Accuracy) na zbiorze testowym
    test_acc = accuracy_score(y_test, y_pred)
    
    # Zapisujemy zebrane wyniki do słownika
    results[name] = {
        "CV Mean Accuracy": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test Accuracy": test_acc,
        "Train Time (s)": train_time,
        # Green IT: wskaźnik efektywności = dokładność / czas (im więcej, tym lepiej)
        "Efficiency (Acc/Time)": test_acc / (train_time if train_time > 0 else 1e-6)
    }

# Konwertujemy wyniki do czytelnej tabeli DataFrame
df_results = pd.DataFrame(results).T

# Sortujemy tabelę według dokładności na zbiorze testowym
df_results = df_results.sort_values(by="Test Accuracy", ascending=False)

# Wyświetlamy gotową tabelę wyników
print("--- ZESTAWIENIE WYNIKÓW MODELI ---")
display(df_results)

## 9. Wybór najlepszego modelu i analiza błędów (Confusion Matrix)

In [ ]:
# Pobieramy nazwę modelu o najwyższej dokładności na zbiorze testowym
best_model_name = df_results.index[0]
best_pipe = models[best_model_name]

print(f"Najlepszy wybrany model: {best_model_name}")
print(f"Dokładność na zbiorze testowym: {df_results.loc[best_model_name, 'Test Accuracy']:.4f}\n")

# Dokonujemy predykcji najlepszym modelem
y_pred_best = best_pipe.predict(X_test)

# Wyświetlamy pełny raport klasyfikacji
print("Raport klasyfikacji:")
print(classification_report(y_test, y_pred_best, target_names=['Złośliwy (0)', 'Łagodny (1)']))

# Obliczamy macierz pomyłek (Confusion Matrix)
cm = confusion_matrix(y_test, y_pred_best)

# Rysujemy macierz pomyłek w formie mapy ciepła (heatmap)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=['Pred: Złośliwy', 'Pred: Łagodny'], 
    yticklabels=['Fakt: Złośliwy', 'Fakt: Łagodny']
)
plt.title(f'Macierz pomyłek - {best_model_name}')
plt.ylabel('Rzeczywista klasa')
plt.xlabel('Przewidziana klasa')
plt.show()

## 10. Ważność cech (Feature Importance)

In [ ]:
# Sprawdzamy, czy najlepszy model posiada współczynniki liniowe lub ważność cech
plt.figure(figsize=(10, 6))

if hasattr(best_pipe.named_steps['clf'], 'feature_importances_'):
    # Dla modeli drzewiastych pobieramy feature_importances_
    importances = best_pipe.named_steps['clf'].feature_importances_
    feature_series = pd.Series(importances, index=X.columns).sort_values(ascending=False)[:10]
    sns.barplot(x=feature_series.values, y=feature_series.index, palette='viridis')
    plt.title('Top 10 najważniejszych cech (Feature Importance)')
    plt.xlabel('Waga istotności')
elif hasattr(best_pipe.named_steps['clf'], 'coef_'):
    # Dla regresji logistycznej pobieramy wartości bezwzględne współczynników wagowych
    importances = np.abs(best_pipe.named_steps['clf'].coef_[0])
    feature_series = pd.Series(importances, index=X.columns).sort_values(ascending=False)[:10]
    sns.barplot(x=feature_series.values, y=feature_series.index, hue=feature_series.index, palette='magma', legend=False)
    plt.title('Top 10 cech o największym wpływie (Współczynniki Regresji)')
    plt.xlabel('Wartość bezwzględna współczynnika')

plt.show()

## 11. Podsumowanie Green IT

In [ ]:
# Wyciągamy model o najwyższej efektywności obliczeniowej
most_efficient_model = df_results.sort_values(by="Efficiency (Acc/Time)", ascending=False).index[0]

print("=== PODSUMOWANIE ASPEKTU GREEN IT ===")
print(f"1. Model najdokładniejszy: {best_model_name} (Accuracy: {df_results.loc[best_model_name, 'Test Accuracy']:.4f})")
print(f"2. Model najbardziej energooszczędny: {most_efficient_model}")
print(f"3. Czas treningu najlepszego modelu: {df_results.loc[best_model_name, 'Train Time (s)']:.4f} s")
print("\nWnioski środowiskowe:")
print("- Prostsze modele (np. Logistic Regression) oferują zbliżoną jakość predykcji przy wielokrotnie niższym koszcie CPU.")
print("- W środowisku produkcyjnym o dużej skali zapytań wybór modelu szybszego pozwala zaoszczędzić energię serwerów.")

## 12. Generowanie struktury folderów, zapis modelu i danych

In [ ]:
# Tworzymy listę katalogów wymaganych przez dokument wytycznych kursu
directories = [
    "data/raw",
    "data/processed",
    "notebooks",
    "src",
    "api",
    "models",
    "reports/figures",
    "config"
]

# Tworzymy każdy folder w bieżącym katalogu roboczym
for dir_path in directories:
    os.makedirs(dir_path, exist_ok=True)
    print(f"Utworzono katalog: {dir_path}")

# Zapisujemy surowe dane do pliku CSV w katalogu data/raw/
df.to_csv("data/raw/breast_cancer.csv", index=False)
print("Zapisano dane surowe do: data/raw/breast_cancer.csv")

# Serializujemy i zapisujemy najlepszy model na dysku do models/best_model.pkl
joblib.dump(best_pipe, "models/best_model.pkl")
print("Zapisano model do: models/best_model.pkl")

# Tworzymy metadane modelu w formacie JSON
metadata = {
    "model_name": best_model_name,
    "test_accuracy": float(df_results.loc[best_model_name, 'Test Accuracy']),
    "random_state": RANDOM_STATE,
    "autor": "Krzysztof Ledworowski"
}

# Zapisujemy plik metadanych na dysku
with open("models/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
print("Zapisano metadane do: models/metadata.json")

## 13. Automatyczne utworzenie modułu `src/features.py`

In [ ]:
%%writefile src/features.py
# Moduł odpowiedzialny za przetwarzanie cech wejściowych
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def get_preprocessing_pipeline():
    """Zwraca potok przygotowujący dane numeryczne do modelu."""
    pipeline = Pipeline([
        ('scaler', StandardScaler())
    ])
    return pipeline

## 14. Automatyczne utworzenie aplikacji `api/app.py` (FastAPI)

In [ ]:
%%writefile api/app.py
# Serwis REST API zbudowany w oparciu o framework FastAPI
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import List

# Inicjalizujemy instancję aplikacji FastAPI
app = FastAPI(
    title="Breast Cancer Detection API",
    description="Endpoint predykcyjny do klasyfikacji nowotworów piersi.",
    version="1.0.0"
)

# Wczytujemy zapisany wcześniej model
model = joblib.load("models/best_model.pkl")

# Definiujemy schemat walidacyjny danych wejściowych za pomocą biblioteki Pydantic
class CancerInput(BaseModel):
    # Oczekujemy listy dokładnie 30 wartości numerycznych odpowiadających cechom guza
    features: List[float] = Field(
        ..., 
        min_items=30, 
        max_items=30, 
        description="Lista 30 cech pomiarowych jądra komórkowego"
    )

# Endpoint sprawdzający stan zdrowia usługi
@app.get("/health")
def health_check():
    return {"status": "healthy", "model_loaded": True}

# Endpoint dokonujący predykcji
@app.post("/predict")
def predict_cancer(data: CancerInput):
    # Konwertujemy otrzymaną listę liczb na wektor dwuwymiarowy
    input_array = np.array(data.features).reshape(1, -1)
    
    # Wykonujemy predykcję za pomocą modelu
    prediction = int(model.predict(input_array)[0])
    
    # Tłumaczymy wynik binarny na etykietę zrozumiałą dla człowieka
    label = "Łagodny (Benign)" if prediction == 1 else "Złośliwy (Malignant)"
    
    # Zwracamy wynik w formacie JSON
    return {
        "prediction_code": prediction,
        "prediction_label": label
    }

## 15. Automatyczne utworzenie `requirements.txt` oraz `Dockerfile`

In [ ]:
%%writefile requirements.txt
fastapi>=0.100.0
uvicorn>=0.22.0
scikit-learn>=1.2.0
pandas>=2.0.0
numpy>=1.24.0
joblib>=1.3.0
pydantic>=2.0.0

In [ ]:
%%writefile Dockerfile
# Używamy lekkiego, energooszczędnego obrazu Pythona (założenie Green IT)
FROM python:3.11-slim

# Ustawiamy katalog roboczy wewnątrz kontenera
WORKDIR /app

# Kopiujemy listę wymaganych bibliotek
COPY requirements.txt .

# Instalujemy pakiety bez przechowywania zbędnego cache'u
RUN pip install --no-cache-dir -r requirements.txt

# Kopiujemy kod projektu i wytrenowany model do kontenera
COPY api/ ./api/
COPY models/ ./models/

# Otwieramy port 8000 dla zapytań HTTP
EXPOSE 8000

# Uruchamiamy aplikację za pomocą serwera produkcyjnego Uvicorn
CMD ["uvicorn", "api.app:app", "--host", "0.0.0.0", "--port", "8000"]

## 16. Automatyczne wygenerowanie `README.md`

In [ ]:
readme_content = f"""# Diagnostyka nowotworów z danych medycznych (Breast Cancer Wisconsin)

## Autor
**Krzysztof Ledworowski**

## Opis projektu
Projekt dyplomowy Data Science realizujący klasyfikację guzów piersi na łagodne i złośliwe na podstawie 30 cech pomiarowych jądra komórkowego.

## Problem biznesowo-medyczny
Celem jest wsparcie diagnostyki wczesnego wykrywania nowotworów przy zachowaniu wysokiej ogólnej dokładności (Accuracy > 90%).

## Dane
- Zbiór: Breast Cancer Wisconsin Diagnostic Dataset (scikit-learn)
- Liczba próbek: 569
- Liczba cech: 30 zmiennych numerycznych ciągłych
- Brakujące wartości: brak (zbiór kompletny)

## Wyniki modeli
- Najlepszy model: **{best_model_name}**
- Dokładność na zbiorze testowym: **{df_results.loc[best_model_name, 'Test Accuracy']:.4f}**

## Green IT
- Zastosowano lekki obraz bazowy `python:3.11-slim` w celu ograniczenia zużycia pamięci i energii serwera.
- Przetestowano czas treningu i efektywność energetyczną algorytmów.

## Instrukcja uruchomienia

### 1. Uruchomienie lokalnego API:
```bash
pip install -r requirements.txt
uvicorn api.app:app --reload --port 8000
```
Dokumentacja Swagger dostępna pod adresem: `http://localhost:8000/docs`

### 2. Budowa i uruchomienie w kontenerze Docker:
```bash
docker build -t breast-cancer-api .
docker run -p 8000:8000 breast-cancer-api
```
"""

# Zapisujemy zawartość do pliku README.md
with open("README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)
    
print("Wygenerowano kompletny plik README.md")

## 17. Konspekt prezentacji wyników (12 slajdów)

In [ ]:
%%writefile reports/prezentacja_slajdy.md
# PREZENTACJA PROJEKTU DYPLOMOWEGO
## Autor: Krzysztof Ledworowski

### Slajd 1: Tytuł
- Temat: Diagnostyka nowotworów piersi z danych medycznych
- Autor: Krzysztof Ledworowski
- Kurs: Data Scientist Online + AI

### Slajd 2: Problem biznesowy i medyczny
- Wczesne wykrywanie zmian nowotworowych ratuje życie
- Cel: automatyczna i szybka klasyfikacja typu guza
- Metryka sukcesu: Accuracy > 90%

### Slajd 3: Dane (Dataset)
- Źródło: Wisconsin Diagnostic Breast Cancer (WDBC)
- 569 próbek pacjentów, 30 cech jądra komórkowego
- Brak braków danych, rozkład: 357 łagodnych, 212 złośliwych

### Slajd 4: Eksploracja danych (EDA)
- Silne korelacje cech geometrycznych (promień, obwód, pole powierzchni)
- Wyraźna separowalność klas w przestrzeni cech wklęsłości

### Slajd 5: Feature Engineering i Pipeline
- Standaryzacja cech za pomocą StandardScaler
- Wykorzystanie potoku `sklearn.pipeline.Pipeline` w celu uniknięcia Data Leakage

### Slajd 6: Modele poddane ewaluacji
- Punkt odniesienia: DummyClassifier (Baseline)
- Modele: Logistic Regression, Random Forest, Gradient Boosting
- Walidacja: 5-Fold Stratified Cross-Validation

### Slajd 7: Wyniki i porównanie
- Prezentacja tabeli z wynikami metryk (CV Accuracy vs Test Accuracy)
- Wyłonienie zwycięskiego modelu

### Slajd 8: Analiza błędów i macierz pomyłek
- Omówienie macierzy pomyłek (Confusion Matrix)
- Analiza przypadków False Negative i False Positive

### Slajd 9: Interpretowalność modelu
- Analiza ważności cech (Feature Importance)
- Najistotniejsze wskaźniki diagnostyczne wpływające na decyzję modelu

### Slajd 10: Wdrożenie (Deployment)
- REST API zbudowane przy użyciu FastAPI
- Walidacja danych wejściowych w Pydantic
- Konteneryzacja w Dockerze

### Slajd 11: Aspekt Green IT
- Porównanie zużycia czasu procesora między algorytmami
- Wybór zoptymalizowanego obrazu `python:3.11-slim`

### Slajd 12: Wnioski i podsumowanie
- Założenia projektu zostały spełnione (Accuracy > 90%)
- Model gotowy do działania w kontenerze

In [ ]:
print("Projekt zakończony pomyślnie! Wygenerowano pliki i strukturę projektu.")